In [ ]:
# ============================================================
# HRSID COCO JSON -> YOLO format
# Image copy 없이 labels만 생성해서 YOLOv8 학습
# ============================================================

!pip install -q ultralytics

import json
import shutil
from pathlib import Path
from collections import defaultdict
from ultralytics import YOLO
from google.colab import drive

drive.mount("/content/drive")

# ============================================================
# 1. Path setting
# ============================================================

BASE_DIR = Path("/content/drive/MyDrive/SAR_AI_Ship_Detection/HRSID")

IMG_DIR = BASE_DIR / "images"
ANN_DIR = BASE_DIR / "annotations"

TRAIN_JSON = ANN_DIR / "train2017.json"
VAL_JSON   = ANN_DIR / "test2017.json"

# YOLO 라벨만 저장할 폴더
YOLO_DIR = BASE_DIR / "HRSID-YOLO"
LABEL_DIR = BASE_DIR / "labels"

MODEL_SAVE_DIR = Path("/content/drive/MyDrive/SAR_AI_Ship_Detection/trained_models")
MODEL_SAVE_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# 2. COCO JSON -> YOLO labels
# ============================================================

def coco_to_yolo_labels(json_path, split_name):
    with open(json_path, "r") as f:
        coco = json.load(f)

    label_out = LABEL_DIR
    label_out.mkdir(parents=True, exist_ok=True)

    images = {img["id"]: img for img in coco["images"]}

    ann_by_img = defaultdict(list)
    for ann in coco["annotations"]:
        ann_by_img[ann["image_id"]].append(ann)

    image_list_path = YOLO_DIR / f"{split_name}.txt"

    converted_images = 0
    missing_images = 0
    total_boxes = 0

    with open(image_list_path, "w") as list_f:

        for img_id, img_info in images.items():
            file_name = img_info["file_name"]
            w = img_info["width"]
            h = img_info["height"]

            img_path = IMG_DIR / file_name

            if not img_path.exists():
                missing_images += 1
                continue

            # 이미지 경로를 txt에 기록
            list_f.write(str(img_path) + "\n")

            yolo_lines = []

            for ann in ann_by_img.get(img_id, []):
                x, y, bw, bh = ann["bbox"]

                if bw <= 0 or bh <= 0:
                    continue

                x_center = (x + bw / 2) / w
                y_center = (y + bh / 2) / h
                bw_norm = bw / w
                bh_norm = bh / h

                cls_id = 0

                yolo_lines.append(
                    f"{cls_id} {x_center:.6f} {y_center:.6f} {bw_norm:.6f} {bh_norm:.6f}"
                )

            label_path = label_out / f"{Path(file_name).stem}.txt"

            with open(label_path, "w") as f:
                f.write("\n".join(yolo_lines))

            converted_images += 1
            total_boxes += len(yolo_lines)

    print(f"[{split_name}] images: {converted_images}")
    print(f"[{split_name}] boxes : {total_boxes}")
    print(f"[{split_name}] missing images: {missing_images}")
    print(f"[{split_name}] image list:", image_list_path)


# 기존 label 폴더 삭제 후 재생성
if LABEL_DIR.exists():
    shutil.rmtree(LABEL_DIR)

YOLO_DIR.mkdir(parents=True, exist_ok=True)

coco_to_yolo_labels(TRAIN_JSON, "train")
coco_to_yolo_labels(VAL_JSON, "val")

# ============================================================
# 3. data.yaml 생성
# ============================================================

data_yaml = YOLO_DIR / "data.yaml"

with open(data_yaml, "w") as f:
    f.write(f"""
path: {YOLO_DIR}

train: train.txt
val: val.txt

names:
  0: ship
""")

print(data_yaml.read_text())

# ============================================================
# 4. Train YOLO
# ============================================================

model = YOLO("yolov8s.pt")

model.train(
    data=str(data_yaml),
    epochs=20,
    imgsz=800,
    batch=8,
    device=0,
    workers=2,
    project="/content/runs",
    name="hrsid_yolov8s"
)

# ============================================================
# 5. Validation
# ============================================================

metrics = model.val(
    data=str(data_yaml),
    split="val",
    imgsz=800,
    batch=8,
    device=0,
    project="/content/eval_runs",
    name="hrsid_yolov8s_val"
)

print("Precision :", metrics.box.mp)
print("Recall    :", metrics.box.mr)
print("mAP50     :", metrics.box.map50)
print("mAP50-95  :", metrics.box.map)

# ============================================================
# 6. Save model to Drive
# ============================================================

shutil.copy(
    "/content/runs/hrsid_yolov8s/weights/best.pt",
    MODEL_SAVE_DIR / "hrsid_yolov8s_best.pt"
)

shutil.copy(
    "/content/runs/hrsid_yolov8s/weights/last.pt",
    MODEL_SAVE_DIR / "hrsid_yolov8s_last.pt"
)

print("Saved:")
print(MODEL_SAVE_DIR / "hrsid_yolov8s_best.pt")
print(MODEL_SAVE_DIR / "hrsid_yolov8s_last.pt")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[train] images: 3642
[train] boxes : 11047
[train] missing images: 0
[train] image list: /content/drive/MyDrive/SAR_AI_Ship_Detection/HRSID/HRSID-YOLO/train.txt
[val] images: 1962
[val] boxes : 5922
[val] missing images: 0
[val] image list: /content/drive/MyDrive/SAR_AI_Ship_Detection/HRSID/HRSID-YOLO/val.txt

path: /content/drive/MyDrive/SAR_AI_Ship_Detection/HRSID/HRSID-YOLO

train: train.txt
val: val.txt

names:
  0: ship

Ultralytics 8.4.68 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/SAR_AI_Ship_Detection/HRSID/HRSID-YOLO/d

FileNotFoundError: [Errno 2] No such file or directory: '/content/runs/hrsid_yolov8s/weights/best.pt'

In [ ]:
import shutil
from pathlib import Path

MODEL_SAVE_DIR = Path(
    "/content/drive/MyDrive/SAR_AI_Ship_Detection/trained_models"
)
MODEL_SAVE_DIR.mkdir(parents=True, exist_ok=True)

shutil.copy(
    "/content/runs/hrsid_yolov8s-4/weights/best.pt",
    MODEL_SAVE_DIR / "hrsid_yolov8s_best.pt"
)

shutil.copy(
    "/content/runs/hrsid_yolov8s-4/weights/last.pt",
    MODEL_SAVE_DIR / "hrsid_yolov8s_last.pt"
)

print("Saved.")

Saved.
